# Food-prep model (Verified Kitchen) — training on a free Colab GPU

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Run the cells in order (Shift+Enter). Cell 2 asks you to upload `foodprep_r01_colab.zip`.
3. The last cell downloads `foodprep_result.zip` (trained weights + test metrics). Nothing else leaves this session;
   **Runtime → Disconnect and delete runtime** afterwards removes the uploaded frames.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics==8.4.212

In [ ]:
from google.colab import files
up = files.upload()  # choose ~/vk-data/datasets/foodprep_r01_colab.zip
!unzip -q -o foodprep_r01_colab.zip -d /content && ls /content/foodprep_r01

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo26n.pt")  # COCO-pretrained: transfer learning for a small dataset
model.train(
    data="/content/foodprep_r01/data.yaml", epochs=100, imgsz=640, batch=16, device=0,
    project="/content/runs", name="foodprep_r01", exist_ok=True, patience=25, seed=0, deterministic=True,
    flipud=0.0, fliplr=0.5, degrees=0.0, mosaic=1.0, close_mosaic=10, plots=True,
)

In [ ]:
import json
best = YOLO("/content/runs/foodprep_r01/weights/best.pt")
m = best.val(data="/content/foodprep_r01/data.yaml", split="test", imgsz=640, device=0, plots=True,
             project="/content/runs/foodprep_r01", name="test", exist_ok=True)
per_class = {best.names[int(c)]: {"precision": round(float(m.box.p[i]), 3), "recall": round(float(m.box.r[i]), 3),
             "mAP50": round(float(m.box.ap50[i]), 3), "mAP50-95": round(float(m.box.ap[i]), 3)}
             for i, c in enumerate(m.box.ap_class_index)}
summary = {"split": "test (held-out blocks + recording E02)", "mAP50": round(float(m.box.map50), 3),
           "mAP50-95": round(float(m.box.map), 3), "precision": round(float(m.box.mp), 3),
           "recall": round(float(m.box.mr), 3), "per_class": per_class}
json.dump(summary, open("/content/runs/foodprep_r01/test_metrics.json", "w"), indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
!cd /content/runs && zip -qr /content/foodprep_result.zip foodprep_r01 -x "foodprep_r01/weights/last.pt"
files.download("/content/foodprep_result.zip")